In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from methods import get_files_with_suffix_and_extension
import numpy as np
from pathlib import Path

MODEL_DICT = {"GPT5.2": "gpt", "gemini-3-pro-preview": "gemini", "llama3v3chat-70B": "llama", "Qwen2.5_72B": "qwen"}

METHOD_SETS = {
    "black_box": ["vanilla", "top_k", "self_probing", "multi_step", "CoT", "self_consistency"],
    "white_box": ["SAR", "semantic_energy"],

    "assessor_error_risk": [
        "asr_difficulty+embedding", "asr_difficulty", "asr_embedding", "asr_demands",
        "asr_difficulty+demands", "asr_difficulty+embedding+demands", "asr_embedding+demands"
    ],

    "answerability_risk_adele": [
        "asr_adele_CoT", "asr_adele_multi_step", "asr_adele_self_probing",
        "asr_adele_top_k", "asr_adele_vanilla"
    ],
    "answerability_risk_diff": [
        "asr_diff_CoT", "asr_diff_multi_step", "asr_diff_self_probing",
        "asr_diff_top_k", "asr_diff_vanilla"
    ],
    "answerability_risk_diff+adele": [
        "asr_diff+adele_CoT", "asr_diff+adele_multi_step", "asr_diff+adele_self_probing",
        "asr_diff+adele_top_k", "asr_diff+adele_vanilla"
    ],
    "answerability_risk_emb": [
        "asr_emb_CoT", "asr_emb_multi_step", "asr_emb_self_probing",
        "asr_emb_top_k", "asr_emb_vanilla"
    ],
    "answerability_risk_emb+adele": [
        "asr_emb+adele_CoT", "asr_emb+adele_multi_step", "asr_emb+adele_self_probing",
        "asr_emb+adele_top_k", "asr_emb+adele_vanilla"
    ],
    "answerability_risk_emb+diff": [
        "asr_emb+diff_CoT", "asr_emb+diff_multi_step", "asr_emb+diff_self_probing",
        "asr_emb+diff_top_k", "asr_emb+diff_vanilla"
    ],
    "answerability_risk_emb+diff+adele": [
        "asr_emb+diff+adele_CoT", "asr_emb+diff+adele_multi_step",
        "asr_emb+diff+adele_self_probing", "asr_emb+diff+adele_top_k",
        "asr_emb+diff+adele_vanilla"
    ],
    # "MAIN": [
    #     "asr_difficulty+embedding", "asr_difficulty+embedding+demands", "asr_emb+diff+adele_CoT", 
    #     "asr_emb+diff+adele_self_probing", "CoT", "self_consistency", "SAR"
    # ]
}

def load_dfs_for_methods(input_folder, result_files, methods, df_bins):
    dfs = []
    labels = []

    for file in result_files:
        method_name = Path(file).stem  

        if method_name in methods:   
            print(file)
            df = pd.read_csv(Path(input_folder) / file)
            df = df[["instance_id", "confidence", "outcome"]]
            df = df.merge(df_bins, on="instance_id", how="left")
            del df["instance_id"]
            dfs.append(df)
            labels.append(method_name)

    return dfs, labels

def load_dfs_for_methods_non_alpha(input_folder, result_files, methods, df_bins):
    # Map: method_name -> filename
    file_by_method = {Path(f).stem: f for f in result_files}

    dfs = []
    labels = []

    # Force order exactly as in `methods`
    for method_name in methods:
        file = file_by_method.get(method_name)
        if file is None:
            continue  # or raise if you want strictness

        print(file)
        df = pd.read_csv(Path(input_folder) / file)
        df = df[["instance_id", "confidence", "outcome"]]
        df = df.merge(df_bins, on="instance_id", how="left")
        del df["instance_id"]

        dfs.append(df)
        labels.append(method_name)

    return dfs, labels


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _pct_by_difficulty(df_subset, outcome_order):
    """
    Compute % of each outcome per difficulty_bin.
    Guaranteed to return:
      - index: difficulty_bin (simple Index)
      - columns: outcome_order
    """
    counts = (
        df_subset
        .groupby(["difficulty_bin", "outcome"], observed=False)
        .size()
        .rename("count")
        .reset_index()
    )

    # total per difficulty bin
    totals = counts.groupby("difficulty_bin")["count"].transform("sum")

    counts["pct"] = 100.0 * counts["count"] / totals

    pct = (
        counts
        .pivot(index="difficulty_bin", columns="outcome", values="pct")
        .fillna(0.0)
    )

    for o in outcome_order:
        if o not in pct.columns:
            pct[o] = 0.0

    return pct[outcome_order].sort_index()



def _nonanswer_pct_by_threshold_0_100(
    df,
    confidence_col="confidence",
    outcome_col="outcome",
):
    """
    Non-answer % (REJECT + AVOIDANCE) for thresholds 0..100, using rule:
      - REJECT if confidence < threshold (applies to ALL rows)
      - else keep raw outcome (CORRECT / INCORRECT / AVOIDANCE)
    """
    conf = np.clip(df[confidence_col].to_numpy(dtype=float), 0.0, 100.0)
    raw = df[outcome_col].astype(str).to_numpy()

    n = len(conf)
    if n == 0:
        return pd.Series(dtype=float)

    is_avoid = raw == "AVOIDANCE"
    conf_avoid = conf[is_avoid]
    total_avoid = conf_avoid.size

    bins = np.arange(0, 102, 1)

    counts_all, _ = np.histogram(conf, bins=bins)
    cumsum_all = np.cumsum(counts_all)

    counts_avoid, _ = np.histogram(conf_avoid, bins=bins)
    cumsum_avoid = np.cumsum(counts_avoid)

    thresholds = np.arange(0, 101, dtype=int)

    reject_count = np.zeros_like(thresholds, dtype=int)
    reject_count[1:] = cumsum_all[:-1]

    avoid_below = np.zeros_like(thresholds, dtype=int)
    avoid_below[1:] = cumsum_avoid[:-1]

    avoid_kept = total_avoid - avoid_below

    nonanswer_pct = (reject_count + avoid_kept) / n * 100.0
    return pd.Series(nonanswer_pct, index=thresholds)


def plot_outcome_by_avoidance_ranges(
    dfs,
    file_labels=None,
    avoidance_ranges=None,
    correct_color="#1d3557",
    reject_color="#b8b2e1",
    avoidance_color="#a8dadc",
    incorrect_color="#e63946",
    figsize_height_per_file=4,
    figsize_width_per_range=4,
    model=None,
    task=None,
    output_file=None,
    confidence_col="confidence",
    outcome_col="outcome",
):
    outcome_order = ["CORRECT", "AVOIDANCE", "REJECT", "INCORRECT"]

    if avoidance_ranges is None:
        avoidance_ranges = [(0, 25), (25, 50), (50, 75), (75, 90), (90, 100)]

    midpoints = np.array([(lo + hi) / 2 for lo, hi in avoidance_ranges], dtype=float)

    n_files = len(dfs)
    n_ranges = len(avoidance_ranges)

    if file_labels is None:
        file_labels = [f"File {i+1}" for i in range(n_files)]

    fig, axes = plt.subplots(
        n_files,
        n_ranges,
        figsize=(figsize_width_per_range * n_ranges, figsize_height_per_file * n_files),
        sharey=True,
    )

    if n_files == 1:
        axes = np.array([axes])
    if n_ranges == 1:
        axes = axes[:, np.newaxis]

    all_bins = pd.Index(range(1, 31), name="difficulty_bin")

    def normalize_pct(pct):
        return pct.reindex(all_bins, fill_value=0.0)

    def full_reject_table():
        t = pd.DataFrame(0.0, index=all_bins, columns=outcome_order)
        t["REJECT"] = 100.0
        return t

    for row_idx, (df_raw, label) in enumerate(zip(dfs, file_labels)):
        df = df_raw.copy()

        nonanswer_by_th = _nonanswer_pct_by_threshold_0_100(
            df, confidence_col, outcome_col
        )

        raw_out = df[outcome_col].astype(str)
        conf = df[confidence_col].astype(float)

        pct_tables = [None] * n_ranges
        meta = [None] * n_ranges

        for j, (low, high) in enumerate(avoidance_ranges):
            is_last = (low, high) == avoidance_ranges[-1]
            mask = (
                (nonanswer_by_th >= low)
                & (nonanswer_by_th <= high if is_last else nonanswer_by_th < high)
            )

            if not mask.any():
                continue

            th = int(mask.index[mask][0])
            eff_out = np.where(conf < th, "REJECT", raw_out)

            df_th = df.copy()
            df_th["outcome"] = pd.Categorical(eff_out, categories=outcome_order)

            pct = normalize_pct(_pct_by_difficulty(df_th, outcome_order))
            pct_tables[j] = pct
            meta[j] = (th, float(nonanswer_by_th.loc[th]))

        # Assume final endpoint if missing
        if pct_tables[-1] is None:
            pct_tables[-1] = full_reject_table()
            meta[-1] = None


        # Interpolate missing
        for j in range(n_ranges):
            if pct_tables[j] is not None:
                continue

            # Find nearest available table to the left (or None if none exists)
            left = next(
                (i for i in range(j - 1, -1, -1) if pct_tables[i] is not None),
                None
            )

            # Right anchor is guaranteed to exist (last column is forced to 100% reject if missing)
            right = min(i for i in range(j + 1, n_ranges) if pct_tables[i] is not None)

            if left is None:
                continue
            
            x0, x1 = midpoints[left], midpoints[right]
            x = midpoints[j]
            w = (x - x0) / (x1 - x0)

            interp = (1 - w) * pct_tables[left] + w * pct_tables[right]

            pct_tables[j] = interp

            nonanswer_pct = float((interp["AVOIDANCE"] + interp["REJECT"]).mean())
            meta[j] = {"interpolated": True, "nonanswer": nonanswer_pct}


        # Plot
        for col_idx, (low, high) in enumerate(avoidance_ranges):
            ax = axes[row_idx, col_idx]
            pct = pct_tables[col_idx]

            if pct is None:
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_ylim(0, 100)
                ax.set_title(f"No data\nNon-answer {low}-{high}%", fontsize=12)
                continue

            c = pct["CORRECT"].values
            a = pct["AVOIDANCE"].values
            r = pct["REJECT"].values
            i = pct["INCORRECT"].values

            x = np.arange(len(pct))

            ax.bar(x, c, width=1, color=correct_color)
            ax.bar(x, a, width=1, bottom=c, color=avoidance_color)
            ax.bar(x, r, width=1, bottom=c + a, color=reject_color)
            ax.bar(x, i, width=1, bottom=c + a + r, color=incorrect_color)

            ax.set_ylim(0, 100)
            ax.set_xticks([])
            ax.set_yticks([])

            info = meta[col_idx]

            if info is not None and "interpolated" not in info:
                th, val = info
                ax.set_title(
                    f"th={th}\n"
                    f"Non-answer {low}-{high}% ({val:.1f}%)",
                    fontsize=12 
                )
            elif info is not None and info.get("interpolated", False):
                ax.set_title(
                    f"Interpolated\n"
                    f"Non-answer {low}-{high}% ({info['nonanswer']:.1f}%)",
                    fontsize=12
                )
            else:
                ax.set_title(
                    f"Interpolated\nNon-answer {low}-{high}%",
                    fontsize=12
                )

        axes[row_idx, 0].set_ylabel(f"{label}\n\nProportion", fontsize=14)

    plt.tight_layout()

    target_path = Path(output_file) if output_file is not None else Path("plots") / model / f"{task}_{MODEL_DICT[model]}.pdf"
    target_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(target_path)

    plt.show()


In [ ]:
MODEL_LIST = ["GPT5.2", "Qwen2.5_72B", "llama3v3chat-70B", "gemini-3-pro-preview"]

TASK_LIST = ["addition", "anagram", "geometry", "logic_puzzles", "science", "locality"]

DISTRIBUTION_LIST = ["OOD"]

for model in MODEL_LIST:
    for task in TASK_LIST:
        for distribution in DISTRIBUTION_LIST:
            input_folder = f"./confidence_results/{distribution}/{model}/{task}/"
            result_files = [
                f for f in get_files_with_suffix_and_extension(
                    input_folder, ".csv", suffix="", prefix=""
                )
                if f != "formatted.csv"
            ]

            if not result_files:
                continue

            df_bins = pd.read_csv(f"./task_difficulty/{task}_bins.csv")

            for method_set_name, methods in METHOD_SETS.items():
                dfs, labels = load_dfs_for_methods(
                    input_folder, result_files, methods, df_bins
                )

                if len(dfs) == 0:
                    continue

                plot_outcome_by_avoidance_ranges(
                    dfs,
                    file_labels=labels,
                    avoidance_ranges=[(0,25), (25,50), (50,75), (75,90), (90,100)],
                    model=model,
                    task=task,
                    output_file=(
                        f"plots/{distribution}/{model}/{task}_{MODEL_DICT[model]}_{method_set_name}.pdf"
                    )
                )
